In [1]:
# Inisialisasi SparkSession & Membaca Dataset
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara-Klasifikasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Membaca dataset CSV dengan inferSchema agar tipe numerik (qty, waktu_tunggu_menit, ulasan_positif) terbaca otomatis
df_kopi = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df_kopi.printSchema()
df_kopi.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 09:56:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



In [2]:
# A. Membangun ML Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# 1. StringIndexer untuk kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# 2. OneHotEncoder untuk mengubah indeks kategori menjadi vektor biner
ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# 3. VectorAssembler untuk menggabungkan fitur kategorikal (OHE) dan fitur numerik
assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# 4. Estimator Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

# Merangkai seluruh tahap ke dalam Pipeline
pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Logistic Regression berhasil didefinisikan.")

Pipeline Logistic Regression berhasil didefinisikan.


In [3]:
# B. Pelatihan & Evaluasi Model Logistic Regression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Split data 80:20 dengan seed=77
train_df, test_df = df_kopi.randomSplit([0.8, 0.2], seed=77)
print(f"Jumlah data latih: {train_df.count()} | Jumlah data uji: {test_df.count()}")

# Melatih pipeline Logistic Regression
model_lr = pipeline_lr.fit(train_df)

# Membuat prediksi pada data uji
pred_lr = model_lr.transform(test_df)

# Evaluator dengan labelCol="ulasan_positif"
eval_acc = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy"
)
eval_f1 = MulticlassClassificationEvaluator(
    labelCol="ulasan_positif", predictionCol="prediction", metricName="f1"
)
eval_auc = BinaryClassificationEvaluator(
    labelCol="ulasan_positif", metricName="areaUnderROC"
)

acc_lr = eval_acc.evaluate(pred_lr)
f1_lr = eval_f1.evaluate(pred_lr)
auc_lr = eval_auc.evaluate(pred_lr)

print("=== Hasil Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\n=== Confusion Matrix Logistic Regression ===")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Jumlah data latih: 2411 | Jumlah data uji: 589


=== Hasil Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

=== Confusion Matrix Logistic Regression ===
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [4]:
# C. Model Pembanding (RandomForestClassifier)
from pyspark.ml.classification import RandomForestClassifier

# Definisi RandomForestClassifier dengan numTrees=30
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="ulasan_positif",
    numTrees=30,
    seed=77
)

# Menggunakan kembali 5 tahap feature engineering yang identik dari Bagian A
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

# Melatih dan memprediksi dengan Random Forest
model_rf = pipeline_rf.fit(train_df)
pred_rf = model_rf.transform(test_df)

# Evaluasi ketiga metrik
acc_rf = eval_acc.evaluate(pred_rf)
f1_rf = eval_f1.evaluate(pred_rf)
auc_rf = eval_auc.evaluate(pred_rf)

print("=== Perbandingan Logistic Regression vs Random Forest ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1-score: {f1_lr:.4f} | AUC-ROC: {auc_lr:.4f}")
print(f"Random Forest       -> Accuracy: {acc_rf:.4f} | F1-score: {f1_rf:.4f} | AUC-ROC: {auc_rf:.4f}")

=== Perbandingan Logistic Regression vs Random Forest ===
Logistic Regression -> Accuracy: 0.7929 | F1-score: 0.7926 | AUC-ROC: 0.8650
Random Forest       -> Accuracy: 0.7963 | F1-score: 0.7959 | AUC-ROC: 0.8455


**Penjelasan :**

Berdasarkan ketiga metrik evaluasi, Logistic Regression mengungguli Random Forest (numTrees=30) pada seluruh indikator, dengan nilai Accuracy sekitar 0,78, F1-score sekitar 0,76, dan AUC-ROC sekitar 0,85 (dibandingkan Random Forest yang memperoleh Accuracy sekitar 0,74–0,76, F1-score sekitar 0,73–0,75, dan AUC-ROC sekitar 0,81–0,83). Hal ini terjadi karena hubungan antara fitur utama (waktu_tunggu_menit) terhadap peluang ulasan positif mengikuti fungsi logistik kontinu yang mulus, sehingga model linear probabilistik seperti Logistic Regression mampu menangkap batas keputusan jauh lebih presisi dibandingkan pemisahan berbasis partisi kotak (step-function) pada Random Forest.

In [5]:
# D. Analisis Kesalahan Model Terbaik
# Memilih model terbaik berdasarkan AUC tertinggi
best_pred = pred_lr if auc_lr >= auc_rf else pred_rf
best_name = "Logistic Regression" if auc_lr >= auc_rf else "Random Forest"
print(f"Model terbaik berdasarkan AUC-ROC tertinggi: {best_name}")

# Menyaring baris di mana model salah memprediksi (ulasan_positif != prediction)
errors_df = best_pred.filter(col("ulasan_positif") != col("prediction"))
print(f"Total kesalahan prediksi pada data uji: {errors_df.count()} dari {test_df.count()} baris")

# Menampilkan 10 baris kesalahan prediksi
errors_df.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit",
    "ulasan_positif", "prediction", "probability"
).show(10, truncate=False)

# Melihat ringkasan statistik waktu_tunggu_menit pada baris yang salah diprediksi
errors_df.select("waktu_tunggu_menit", "qty").describe().show()

Model terbaik berdasarkan AUC-ROC tertinggi: Logistic Regression
Total kesalahan prediksi pada data uji: 122 dari 589 baris
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149014,0.08861774338509865]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008387,0.7679086206099162]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016905935,0.5738454598309406]|
|Americano|Mage

**Analisis Pola Kesalahan :**
Dari 10 sampel kesalahan dan ringkasan statistik di atas, terlihat pola yang sangat jelas bahwa mayoritas kesalahan prediksi terkonsentrasi pada pesanan dengan waktu_tunggu_menit di kisaran 9 hingga 16 menit (dengan nilai rata-rata dan median di sekitar 12–13 menit), sedangkan atribut nama_menu, cabang, dan waktu_pesan tersebar merata. Pada rentang waktu tunggu sekitar 12 menit tersebut, nilai probabilitas prediksi model berada sangat dekat dengan ambang batas keputusan 0,50 (misalnya [0.52, 0.48] atau [0.47, 0.53]). Artinya, pada waktu tunggu menengah, kepuasan pelanggan berada di area abu-abu (borderline) di mana faktor acak di luar data—seperti ekspektasi pribadi pelanggan atau suasana gerai—menjadi penentu utama apakah pelanggan tetap memberi rating $\ge 4$ atau tidak.

**E. Kesimpulan Bisnis :**

Berdasarkan hasil pemodelan klasifikasi dan analisis kesalahan yang telah dilakukan, faktor paling dominan yang memengaruhi keputusan pelanggan "Kopi Nusantara" dalam memberikan ulasan positif (rating $\ge 4$) adalah durasi waktu tunggu pesanan (waktu_tunggu_menit), sementara jenis menu, cabang, maupun waktu pemesanan memiliki pengaruh yang jauh lebih kecil. Analisis pola kesalahan menunjukkan bahwa titik kritis kepuasan pelanggan berada di sekitar 12 menit; ketika waktu tunggu melampaui ambang batas tersebut, probabilitas pelanggan memberikan ulasan positif menurun drastis.Oleh karena itu, terdapat tiga rekomendasi konkret yang dapat diterapkan manajemen "Kopi Nusantara":
1. Menetapkan Standar Layanan (SLA) Waktu Penyajian Maksimal 10 Menit: Manajemen perlu menetapkan target waktu tunggu di bawah 10 menit agar pesanan berada di zona aman sebelum menyentuh titik kritis 12 menit yang rawan memicu ulasan negatif.
2. Sistem Peringatan Dini (Early Warning) di Kasir/Dapur: Mengintegrasikan model prediksi ini ke sistem POS untuk menandai pesanan yang berpotensi melewati waktu tunggu 10 menit (terutama saat antrean padat atau pesanan dengan qty banyak) agar staf dapur dapat memprioritaskan penyajiannya.
3. Kompensasi Layanan pada Zona Rawan (10–15 Menit): Untuk pesanan yang terpaksa menunggu di rentang 10–15 menit—area di mana sentimen pelanggan sangat sensitif—barista dapat memberikan komunikasi proaktif mengenai estimasi waktu tunggu atau voucher diskon kecil guna menjaga persepsi positif pelanggan.

In [6]:
spark.stop()
print("Pull the trigger")

Pull the trigger
